In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('variational_ae.py'), '..')))

from tensorflow.keras.datasets import mnist
from variational_ae import VariationalAutoencoder
import numpy as np
from sklearn.model_selection import train_test_split
import h5py

In [ ]:
with h5py.File('Dataset/log_spec_data_dataset.h5', 'r') as h5f:
    log_spec_data_train = h5f['train'][:]
    log_spec_data_labels = h5f['label'][:]

In [ ]:
with h5py.File('Dataset/log_mel_spec_data_dataset.h5', 'r') as h5f:
    log_mel_spec_data_train = h5f['train'][:]
    log_mel_spec_data_labels = h5f['label'][:]

In [ ]:
log_spec_data_train = log_spec_data_train[..., np.newaxis]
log_mel_spec_data_train = log_mel_spec_data_train[..., np.newaxis]

log_spec_x_train, log_spec_x_val = train_test_split(log_spec_data_train, test_size=0.05, random_state=42)
log_mel_x_train, log_mel_x_val = train_test_split(log_mel_spec_data_train, test_size=0.05, random_state=42)

In [ ]:
print("Log Spec Train shape:", log_spec_x_train.shape)
print("Log Spec Validation shape:", log_spec_x_val.shape)

print("Log Mel Train shape:", log_spec_x_train.shape)
print("Log Mel Validation shape:", log_spec_x_val.shape)

In [ ]:
LEARNING_RATE = 0.0005
BATCH_SIZE = 32
EPOCHS = 50

In [ ]:
input_shape = log_spec_x_train.shape[1:]
latent_space_dim = 2
decoder_out_filter = 1

In [ ]:
# Hyperparameters for the Variational Autoencoder
recon_weight = 1000.0  # Weight for the reconstruction loss.
beta = 1.0  # Weight for the KL divergence loss.

In [ ]:
autoencoder = VariationalAutoencoder(input_shape, latent_space_dim, decoder_out_filter, recon_weight, beta, conv_layers_config=[
    {'filters': 32, 'kernel_size': (3, 3), 'strides': (1, 1)},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': (2, 2)},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': (2, 2)},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': (1, 1)}
])

In [ ]:
autoencoder.compile(learning_rate=LEARNING_RATE)

In [ ]:
autoencoder.summary()

In [ ]:
autoencoder.fit(
    x=log_spec_x_train,
    y={"reconstruction": log_spec_x_train}, # Autoencoders typically use the same data for input and output
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(log_spec_x_val, {"reconstruction": log_spec_x_val}), # Validation data for monitoring
    shuffle=True
)

In [ ]:
autoencoder.save_all()  # Save the trained model